# Evaluate Boolean vs Vector Space Retrieval

This notebook replaces `src/evaluation/evaluation.py`.

In [ ]:
import json
import logging
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import pandas as pd
import requests
from matplotlib.ticker import MaxNLocator

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("evaluation-notebook")

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    project_root = cwd

# Ensure config paths resolve against the API project root, not notebooks/.
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))

from config import (
    GROUND_DATASET_END_DATE,
    GROUND_DATASET_START_DATE,
)

# Use project-root anchored paths to avoid cwd-related path issues in notebooks.
files_dir = project_root / "files"
GROUND_DATASET_FILE_PATH = str(files_dir / "ground_truth.csv")
EVAL_MEASURES_CSV_PATH = str(files_dir / "evaluation_measures.csv")
EVAL_MEASURES_IMAGE_PATH = str(files_dir / "evaluation_measures.png")
EVAL_TEMP_RELEVANCE_IMAGE_PATH = str(files_dir / "evaluation_temporal_relevance.png")

base_url = os.getenv("IR_API_BASE_URL", "http://127.0.0.1:8000")
boolean_api_url = f"{base_url}/search/boolean"
vector_space_url = f"{base_url}/search/vector-space"

vector_queries = [
    "political corruption scandals",
    "election 2024 turnout",
    "democratic party",
    "women rights people power party",
]

boolean_queries = [
    [
        {"operator": "AND", "value": "political"},
        {"operator": "AND", "value": "corruption"},
        {"operator": "OR", "value": "scandals"},
    ],
    [
        {"operator": "AND", "value": "election"},
        {"operator": "OR", "value": "2024"},
        {"operator": "OR", "value": "turnout"},
    ],
    [
        {"operator": "AND", "value": "democratic"},
        {"operator": "OR", "value": "party"},
    ],
    [
        {"operator": "AND", "value": "women"},
        {"operator": "AND", "value": "rights"},
        {"operator": "OR", "value": "people"},
        {"operator": "OR", "value": "power"},
        {"operator": "AND", "value": "party"},
    ],
]

In [ ]:
def evaluate_search_model(relevant_docs: List[int], retrieved_docs: List[int]) -> tuple[float, float, float]:
    relevant_retrieved_docs = set(relevant_docs).intersection(set(retrieved_docs))
    true_positives = len(relevant_retrieved_docs)
    false_positives = len(retrieved_docs) - true_positives
    false_negatives = len(relevant_docs) - true_positives
    recall = true_positives / (true_positives + false_negatives) if relevant_docs else 0
    precision = true_positives / (true_positives + false_positives) if retrieved_docs else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return recall, precision, f1

def call_boolean_api(query) -> List[dict]:
    headers = {"Content-Type": "application/json"}
    response = requests.post(boolean_api_url, data=json.dumps(query), headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()

def call_vector_space_api(query: str) -> List[dict]:
    query_string = "+".join(query.split())
    response = requests.get(f"{vector_space_url}?q={query_string}", timeout=30)
    response.raise_for_status()
    return response.json()

def calculate_temporal_relevance(retrieved_docs: List[dict], start_date: datetime, end_date: datetime) -> dict:
    date_counts = {str(start_date + timedelta(days=i)): 0 for i in range((end_date - start_date).days + 1)}
    for doc in retrieved_docs:
        date_time_obj = datetime.strptime(doc["published_on"], "%Y-%m-%dT%H:%M:%SZ")
        date_key = date_time_obj.replace(second=0, minute=0, hour=0).strftime("%Y-%m-%d %H:%M:%S")
        if date_key in date_counts:
            date_counts[date_key] += 1
    return {datetime.strptime(date, "%Y-%m-%d %H:%M:%S").date(): count for date, count in date_counts.items()}

def get_relevant_docs(query: str, ground_truth_df: pd.DataFrame) -> List[int]:
    relevant_docs = ground_truth_df[[query, "id"]]
    return relevant_docs[relevant_docs[query] == True]["id"].tolist()

def plot_evaluation_results(results: pd.DataFrame):
    graph_data = results.drop(columns=["boolean_temporal_relevance", "vector_space_temporal_relevance"])
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
    axes = graph_data.set_index("query").plot(kind="bar", subplots=True, ax=axes, layout=(2, 3), legend=False)
    for ax in axes.flatten():
        ax.set_ylim([0, 1])
        ax.set_ylabel("Score")
        ax.set_xticks(range(len(graph_data)))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment="right")
    plt.tight_layout()
    plt.savefig(EVAL_MEASURES_IMAGE_PATH)
    plt.show()

def plot_temporal_relevance(results: pd.DataFrame):
    num_results = len(results)
    num_cols = 2
    num_rows = num_results // num_cols + (num_results % num_cols > 0)
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(10, 5 * num_rows))
    axs_flat = axs.flatten() if hasattr(axs, "flatten") else [axs]
    for i, (_, row) in enumerate(results.iterrows()):
        b_rel = row["boolean_temporal_relevance"]
        v_rel = row["vector_space_temporal_relevance"]
        dates = list(b_rel.keys())
        counts_boolean = list(b_rel.values())
        counts_vector = list(v_rel.values())
        ax = axs_flat[i]
        width = 0.4
        ax.bar([x - width / 2 for x in range(len(dates))], counts_boolean, width, label="Boolean", color="blue", alpha=0.5)
        ax.bar([x + width / 2 for x in range(len(dates))], counts_vector, width, label="Vector Space", color="orange", alpha=0.5)
        ax.set_xlabel("Date")
        ax.set_ylabel("Number of Documents")
        ax.set_title(f"query: {row['query']}")
        ax.set_xticks(range(len(dates)))
        ax.set_xticklabels(dates, rotation=45, horizontalalignment="right")
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        ax.legend()
    for j in range(num_results, len(axs_flat)):
        fig.delaxes(axs_flat[j])
    plt.tight_layout()
    plt.savefig(EVAL_TEMP_RELEVANCE_IMAGE_PATH)
    plt.show()

In [ ]:
if not GROUND_DATASET_START_DATE or not GROUND_DATASET_END_DATE:
    raise ValueError("GROUND_DATASET_START_DATE and GROUND_DATASET_END_DATE must be set in .env")

ground_truth_df = pd.read_csv(GROUND_DATASET_FILE_PATH)
results = []
for idx in range(len(vector_queries)):
    relevant_docs = get_relevant_docs(vector_queries[idx], ground_truth_df)
    boolean_response = call_boolean_api(boolean_queries[idx])
    vector_response = call_vector_space_api(vector_queries[idx])
    boolean_docs = sorted([int(doc["id"]) for doc in boolean_response])
    vector_docs = sorted([int(doc["id"]) for doc in vector_response])
    boolean_recall, boolean_precision, boolean_f1 = evaluate_search_model(relevant_docs, boolean_docs)
    vector_recall, vector_precision, vector_f1 = evaluate_search_model(relevant_docs, vector_docs)
    results.append({
        "query": vector_queries[idx],
        "boolean_recall": boolean_recall,
        "boolean_precision": boolean_precision,
        "boolean_f1": boolean_f1,
        "boolean_temporal_relevance": calculate_temporal_relevance(boolean_response, GROUND_DATASET_START_DATE, GROUND_DATASET_END_DATE),
        "vector_space_recall": vector_recall,
        "vector_space_precision": vector_precision,
        "vector_space_f1": vector_f1,
        "vector_space_temporal_relevance": calculate_temporal_relevance(vector_response, GROUND_DATASET_START_DATE, GROUND_DATASET_END_DATE),
    })

df = pd.DataFrame(results)
display(df)
df.describe().to_csv(EVAL_MEASURES_CSV_PATH)
plot_evaluation_results(df)
plot_temporal_relevance(df)
print(EVAL_MEASURES_CSV_PATH)
print(EVAL_MEASURES_IMAGE_PATH)
print(EVAL_TEMP_RELEVANCE_IMAGE_PATH)